# 3 Aplicação de modelos de Machine Learning para prever o preço médio dos carros

Nesta etapa serão aplicados modelos de regressão para prever o preço médio dos carros (avg_price_brl).  
Serão utilizados os algoritmos Random Forest Regressor e XGBoost Regressor.

As variáveis categóricas serão transformadas em variáveis numéricas para permitir o treinamento dos modelos.

## Preparação dos dados

Antes da aplicação dos modelos de machine learning, é necessário carregar e preparar a base de dados.

Nesta etapa são realizadas as seguintes operações:

- Carregamento da base `precos_carros_brasil.csv`
- Remoção de linhas com valores faltantes
- Remoção de registros duplicados
- Conversão da variável `engine_size` para formato numérico
- Criação da variável `month_of_reference_num`, que representa os meses em formato numérico

Após essas etapas, o dataframe `dados` será utilizado como base para o treinamento dos modelos de regressão.

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# carregar base
dados = pd.read_csv('precos_carros_brasil.csv')

# remover linhas vazias
dados = dados.dropna()

# remover duplicados
dados.drop_duplicates(inplace=True)

# converter engine_size
dados['engine_size'] = dados['engine_size'].astype(str).str.replace(',', '.').astype(float)

# converter meses para número
meses = {
    'January':1,'February':2,'March':3,'April':4,
    'May':5,'June':6,'July':7,'August':8,
    'September':9,'October':10,'November':11,'December':12
}

dados['month_of_reference_num'] = dados['month_of_reference'].map(meses)

dados.shape

(202295, 12)

## 3a) Seleção das variáveis para o modelo

A variável alvo (target) será **avg_price_brl**.

As variáveis numéricas utilizadas como preditoras serão:

- year_of_reference
- engine_size
- year_model
- month_of_reference_num

As variáveis categóricas **brand**, **fuel** e **gear** serão transformadas em variáveis numéricas utilizando **One Hot Encoding**.

In [2]:
# Importações necessárias
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [3]:
# Variável target
y = dados['avg_price_brl']

# Variáveis independentes
X = dados[['year_of_reference','engine_size','year_model','month_of_reference_num','brand','fuel','gear']]

In [4]:
# Separação entre colunas numéricas e categóricas

numericas = ['year_of_reference','engine_size','year_model','month_of_reference_num']
categoricas = ['brand','fuel','gear']

In [5]:
# One Hot Encoding para variáveis categóricas

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categoricas)
    ],
    remainder='passthrough'
)

## 3b) Divisão dos dados em treino e teste

Os dados foram divididos em:

- 75% para treino
- 25% para teste

In [6]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

## 3c) Treinamento dos modelos

Foram utilizados dois algoritmos de regressão:

- RandomForestRegressor
- XGBRegressor (XGBoost)

Os modelos foram treinados utilizando os dados de treino previamente separados.

In [7]:
# Importando modelos
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [8]:
# Pipeline Random Forest

pipeline_rf = Pipeline(steps=[
    ('prep', preprocessador),
    ('modelo', RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

In [9]:
# Pipeline XGBoost

pipeline_xgb = Pipeline(steps=[
    ('prep', preprocessador),
    ('modelo', XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ))
])

In [10]:
# Treinando os modelos

pipeline_rf.fit(X_treino, y_treino)
pipeline_xgb.fit(X_treino, y_treino)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse 

## 3d) Geração das previsões

Após o treinamento, os modelos foram utilizados para gerar previsões dos preços médios dos carros no conjunto de teste.

In [11]:
# Predições

pred_rf = pipeline_rf.predict(X_teste)
pred_xgb = pipeline_xgb.predict(X_teste)

## 3e) Análise de importância das variáveis

A análise de importância permite identificar quais variáveis tiveram maior impacto na previsão do preço médio dos carros.

In [12]:
import pandas as pd

# Pegando nomes das colunas após encoding

encoder = pipeline_rf.named_steps['prep'].named_transformers_['cat']
nomes_categoricos = encoder.get_feature_names_out(categoricas)

nomes_variaveis = list(nomes_categoricos) + numericas

In [13]:
# Importância Random Forest

importancias_rf = pipeline_rf.named_steps['modelo'].feature_importances_

importancia_rf_df = pd.DataFrame({
    'variavel': nomes_variaveis,
    'importancia': importancias_rf
}).sort_values(by='importancia', ascending=False)

importancia_rf_df.head(10)

,variavel,importancia
12,engine_size,0.423056
13,year_model,0.391407
7,fuel_Diesel,0.074723
10,gear_manual,0.020154
9,gear_automatic,0.019550
11,year_of_reference,0.013723
8,fuel_Gasoline,0.013040
14,month_of_reference_num,0.012368
5,brand_VW - VolksWagen,0.011335
3,brand_Nissan,0.010370


In [14]:
# Importância XGBoost

importancias_xgb = pipeline_xgb.named_steps['modelo'].feature_importances_

importancia_xgb_df = pd.DataFrame({
    'variavel': nomes_variaveis,
    'importancia': importancias_xgb
}).sort_values(by='importancia', ascending=False)

importancia_xgb_df.head(10)

,variavel,importancia
7,fuel_Diesel,0.249878
12,engine_size,0.243261
13,year_model,0.203904
3,brand_Nissan,0.128753
9,gear_automatic,0.052916
4,brand_Renault,0.037764
5,brand_VW - VolksWagen,0.031270
11,year_of_reference,0.016465
0,brand_Fiat,0.015986
2,brand_GM - Chevrolet,0.007738


## 3f) Dê uma breve explicação (máximo de quatro linhas) sobre os resultados encontrados na análise de importância de variáveis

A análise de importância das variáveis indica quais características mais influenciam a previsão do preço médio dos veículos.

Nos dois modelos analisados, destacaram-se principalmente as variáveis `engine_size`, `year_model` e `fuel_Diesel`.

Isso indica que veículos com motores maiores tendem a possuir preços mais elevados. Além disso, o ano do modelo é um fator relevante, pois veículos mais novos geralmente possuem maior valor de mercado. O tipo de combustível também influencia o preço, sendo que veículos a diesel costumam ter valores mais altos em comparação com outros tipos de combustível.

Portanto, essas variáveis são as que mais contribuem para a capacidade preditiva dos modelos utilizados.

## 3g) Avaliação dos modelos

Para avaliar o desempenho dos modelos foram utilizadas as seguintes métricas:

- MSE (Mean Squared Error)
- MAE (Mean Absolute Error)
- R² (Coeficiente de determinação)

In [15]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [16]:
# Random Forest

mse_rf = mean_squared_error(y_teste, pred_rf)
mae_rf = mean_absolute_error(y_teste, pred_rf)
r2_rf = r2_score(y_teste, pred_rf)

print("Random Forest")
print("MSE:", mse_rf)
print("MAE:", mae_rf)
print("R2:", r2_rf)

Random Forest
MSE: 141545606.7453853
MAE: 5858.226771635237
R2: 0.9474053983875719


In [17]:
# XGBoost

mse_xgb = mean_squared_error(y_teste, pred_xgb)
mae_xgb = mean_absolute_error(y_teste, pred_xgb)
r2_xgb = r2_score(y_teste, pred_xgb)

print("XGBoost")
print("MSE:", mse_xgb)
print("MAE:", mae_xgb)
print("R2:", r2_xgb)

XGBoost
MSE: 109882261.70527738
MAE: 5680.408439638342
R2: 0.9591706594676769


## 3h) Dê uma breve explicação (máximo de quatro linhas) sobre qual modelo gerou o melhor resultado e a métrica de avaliação utilizada

Ao comparar os resultados obtidos, observa-se que o modelo XGBoost apresentou melhor desempenho em relação ao Random Forest.

O modelo XGBoost obteve valores menores de MSE e MAE, indicando menor erro médio nas previsões. Além disso, apresentou um valor de R² maior, mostrando maior capacidade de explicar a variação do preço médio dos veículos.

Dessa forma, conclui-se que o modelo XGBoost foi o que apresentou melhor desempenho para o problema de previsão de preços de carros.